# 03. Baseline на правилах

**Цель:** оценить простые правила для определения границ предложений. Зафиксировать baseline F1, который ML-модель должна превзойти.

**Задача:** дан `clean_text` (текст без нумерации) → предсказать позиции границ предложений.

**Данные:** `data/processed/sentences.jsonl` (88 чистых текстов, исключены text_833, text_843)

In [ ]:
import json
import random
from pathlib import Path

DATA_PATH = Path("../data/processed/sentences.jsonl")
EXCLUDE_IDS = {"text_833", "text_843"}

with open(DATA_PATH, encoding="utf-8") as f:
    dataset = [json.loads(line) for line in f]

dataset = [r for r in dataset if r["id"] not in EXCLUDE_IDS]
print(f"Текстов: {len(dataset)}")
print(f"Предложений: {sum(r['num_sentences'] for r in dataset)}")

## 1. Функция оценки

Для каждого правила: предсказываем позиции границ → сравниваем с истинными → считаем Precision, Recall, F1.

Граница = позиция `s["end"]` (конец предложения = начало следующего). Допуск при сравнении: ±2 символа.

In [ ]:
def get_true_boundaries(rec):
    """Истинные границы: позиции end каждого предложения (кроме последнего)."""
    sents = rec["sentences"]
    return set(sents[i]["end"] for i in range(len(sents) - 1))


def evaluate(dataset, predict_fn, tolerance=2):
    """Оценивает predict_fn на всём датасете.

    predict_fn(clean_text) -> set of predicted boundary positions.
    """
    total_tp = total_fp = total_fn = 0

    for rec in dataset:
        true_bd = get_true_boundaries(rec)
        pred_bd = predict_fn(rec["clean_text"])

        matched_true = set()
        matched_pred = set()
        for pb in pred_bd:
            for tb in true_bd:
                if abs(pb - tb) <= tolerance and tb not in matched_true:
                    matched_true.add(tb)
                    matched_pred.add(pb)
                    break

        total_tp += len(matched_pred)
        total_fp += len(pred_bd) - len(matched_pred)
        total_fn += len(true_bd) - len(matched_true)

    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    return {
        "TP": total_tp, "FP": total_fp, "FN": total_fn,
        "Precision": precision, "Recall": recall, "F1": f1,
    }

## 2. Baseline-правила

### Правило 1 (naive): граница после каждого `.!?…`
### Правило 2 (standard): `.!?…` + пробел + заглавная буква
### Правило 3 (extended): standard + тире/кавычки перед заглавной

In [ ]:
EOS = set(".!?\u2026")
DASHES = set("\u2014\u2013\u2212-\u2212")


def baseline_naive(text):
    """Граница после каждого .!?…"""
    boundaries = set()
    for i, ch in enumerate(text):
        if ch in EOS:
            boundaries.add(i + 1)
    return boundaries


def baseline_standard(text):
    """eos_punct + пробел + заглавная буква"""
    boundaries = set()
    for i in range(1, len(text) - 1):
        if text[i - 1] in EOS and text[i] == " " and text[i + 1].isupper():
            boundaries.add(i + 1)
    return boundaries


def baseline_extended(text):
    """standard + тире/кавычки перед заглавной"""
    boundaries = baseline_standard(text)
    for i in range(1, len(text) - 1):
        # тире + пробел + заглавная (или без пробела)
        if text[i - 1] in DASHES and text[i] == " " and text[i + 1].isupper():
            boundaries.add(i + 1)
        if text[i - 1] in DASHES and text[i].isupper():
            boundaries.add(i)
        # « + заглавная (начало цитаты как нового предложения)
        if text[i - 1] == "\u00ab" and text[i].isupper():
            # только если перед « есть пробел или пунктуация
            if i >= 2 and (text[i - 2] == " " or text[i - 2] in EOS):
                boundaries.add(i - 1)
        # »  + пробел + заглавная
        if text[i - 1] == "\u00bb" and text[i] == " " and text[i + 1].isupper():
            boundaries.add(i + 1)
    return boundaries


results = {}
for name, fn in [("naive", baseline_naive),
                 ("standard", baseline_standard),
                 ("extended", baseline_extended)]:
    res = evaluate(dataset, fn)
    results[name] = res
    print(f"=== {name} ===")
    print(f"  TP={res['TP']}, FP={res['FP']}, FN={res['FN']}")
    print(f"  Precision = {res['Precision']:.3f}")
    print(f"  Recall    = {res['Recall']:.3f}")
    print(f"  F1        = {res['F1']:.3f}\n")

## 3. Анализ ошибок лучшего baseline

Какие границы пропускает лучшее правило (FN)? Какие ложные срабатывания (FP)?

In [ ]:
# Выбираем лучший baseline по F1
best_name = max(results, key=lambda k: results[k]["F1"])
best_fn = {"naive": baseline_naive, "standard": baseline_standard, "extended": baseline_extended}[best_name]
print(f"Лучший baseline: {best_name} (F1={results[best_name]['F1']:.3f})\n")

# Собираем FN и FP примеры
fn_examples = []
fp_examples = []

for rec in dataset:
    ct = rec["clean_text"]
    true_bd = get_true_boundaries(rec)
    pred_bd = best_fn(ct)

    matched_true = set()
    matched_pred = set()
    for pb in pred_bd:
        for tb in true_bd:
            if abs(pb - tb) <= 2 and tb not in matched_true:
                matched_true.add(tb)
                matched_pred.add(pb)
                break

    for tb in true_bd - matched_true:
        context = ct[max(0, tb - 25):min(len(ct), tb + 25)]
        fn_examples.append(context)

    for pb in pred_bd - matched_pred:
        context = ct[max(0, pb - 25):min(len(ct), pb + 25)]
        fp_examples.append(context)

random.seed(42)

print(f"Пропущенных границ (FN): {len(fn_examples)}")
print("\nПримеры FN (контекст ±25 символов):")
for ex in random.sample(fn_examples, min(15, len(fn_examples))):
    print(f"  ...{ex}...")

print(f"\nЛожных срабатываний (FP): {len(fp_examples)}")
print("\nПримеры FP (контекст ±25 символов):")
for ex in random.sample(fp_examples, min(15, len(fp_examples))):
    print(f"  ...{ex}...")

## 4. Сводка

| Правило | Precision | Recall | F1 |
|---------|-----------|--------|-----|
| naive (после каждого `.!?…`) | низкая | высокая | низкий |
| standard (eos + пробел + заглавная) | ~0.96 | ~0.86 | ~0.91 |
| extended (+ тире, кавычки) | ? | ? | ? |

Лучший baseline F1 — это порог, который ML-модель в `04_ml_model.ipynb` должна превзойти.